# Enveda v9 submission: FPNet single + 10ppm nocap canonical top100 + ICEBERG F1-split fused_merge top25
Union 712,199 pool, single max-peak spectrum, 10ppm full-window nocap, shipped scoring -> canonical max merge -> top100 shortlist (validated caliber; F1-optimal K), per-record CE-split forward (single-CE original path; multi-CE one prediction per CE; peak cache key model|ckpt|structure|CE|adduct|instrument|prep) -> fused_merge primary (equal-weight merged predictions then similarity) + sims-mean auxiliary (both stored, merge-ranked; missing=None never 0) -> top25 -> RDKit canonical export. Budget: top100 split = 43,707 unique jobs on NP250 at ~26-30/s -> ~25-30 min + load/overhead; forward budget_s=3600; total far below 9h. RDKit from wheelhouse offline (version printed as-is); weights msg_all mirror (NIST excluded).

In [ ]:
import sys
print(sys.version)
import torch
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
assert torch.cuda.is_available(), 'need GPU'
import numpy as np, pyarrow.parquet as pq
print('numpy/pyarrow ok')

In [ ]:
from pathlib import Path
def find(name):
    hits = list(Path('/kaggle/input').rglob(name))
    assert hits, 'missing input: ' + name
    print(name, '->', hits[0])
    return str(hits[0])
TEST = find('test.parquet')
W_SINGLE = find('fp_single_s2.pt')
T_FP = find('fp.npy')
T_MASS = find('mass.npy')
T_KEYS = find('keys.npy')
T_SMI = find('smiles.txt')
C_FP = find('coco_fp.npy')
C_MASS = find('coco_mass.npy')
C_META = find('coco_meta.pkl')
print('torch cuda devices:', torch.cuda.device_count())

In [ ]:
import pickle, time
import numpy as np
t0=time.perf_counter()
t_fp = np.load(T_FP, mmap_mode='r')
t_mass = np.load(T_MASS, mmap_mode='r')
t_keys = np.load(T_KEYS, allow_pickle=True)
t_smi = open(T_SMI).read().splitlines()
print('train pool:', t_fp.shape, len(t_keys), len(t_smi), round(time.perf_counter()-t0,1))
assert t_fp.shape[0]==len(t_keys)==len(t_smi)==len(t_mass)==275810
meta = pickle.load(open(C_META,'rb'))
ckeys = list(meta['keys']); csmi = list(meta['smiles'])
c_mass = np.load(C_MASS, mmap_mode='r')
c_fp = np.load(C_FP, mmap_mode='r')
print('coco pool:', c_fp.shape, len(ckeys))
tset = set(t_keys)
cmap = dict(zip(ckeys, np.asarray(c_mass, float)))
cidx = {k:i for i,k in enumerate(ckeys)}
allk = list(t_keys) + [k for k in ckeys if k not in tset]
alls = t_smi + [s for k,s in zip(ckeys,csmi) if k not in tset]
allm = np.concatenate([np.asarray(t_mass, float), np.array([cmap[k] for k in ckeys if k not in tset], float)])
print('combined:', len(allk), round(time.perf_counter()-t0,1))
TQ = pq.read_table(TEST).to_pydict()
mol = {}
for i in range(len(TQ['molecule_id'])):
    mol.setdefault(TQ['molecule_id'][i], []).append(i)
print('test molecules:', len(mol), 'spectra:', len(TQ['molecule_id']))
assert len(allk)==712199, f'union pool {len(allk)} != 712199'
print('union 712,199 ok')


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math
"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]
INSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = it >= floor*mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N
        order = np.argsort(-it)
        bucket = (mz//win).astype(np.int64)
        cnt = {}; sel=[]
        for i in order:
            b = bucket[i]; c = cnt.get(b,0)
            if c < per_win: cnt[b]=c+1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks-len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it/it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim//2
        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)
        self.register_buffer('inv', (2*math.pi)/wav)
    def forward(self, x):                      # x: (...,)
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h=h
        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))
        self.drop=nn.Dropout(drop)
    def forward(self, x, pad):
        B,N,D=x.shape; y=self.n1(x)
        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)
        m = (~pad)[:,None,None,:]                       # True = attend
        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d=d
        self.mz_emb  = SinEmb(d)
        self.nl_emb  = SinEmb(d)
        self.pk = nn.Linear(2*d+1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d+3, d)
        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B,N = mz.shape
        nl = (prec[:,None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:,0]
        msk = (~pad[:,1:]).float().unsqueeze(-1)
        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))
ck = torch.load(W_SINGLE, map_location='cpu', weights_only=True)
net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().eval()
net.load_state_dict(ck['model'], strict=True)
print('model ok:', ck['nbits'], ck['d'], ck['layers'])
NTR = 275810
assert t_fp.shape == (275810, 867)

In [ ]:
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
    S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
    Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {'[M+H]+':(1,1,PROTON),'[M+NH4]+':(1,1,NH4-E),
 '[M+Na]+':(1,1,MASS['Na']-E),'[M+K]+':(1,1,MASS['K']-E),
 '[M-H2O+H]+':(1,1,PROTON-H2O),'[M-2H2O+H]+':(1,1,PROTON-2*H2O),
 '[M+2H]2+':(1,2,2*PROTON),'[M]+':(1,1,-E),'[M-H2O]+':(1,1,-E-H2O),
 '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 '[M-H]-':(1,1,-PROTON),'[M-H2O-H]-':(1,1,-PROTON-H2O),
 '[M+CH2O2-H]-':(1,1,FORMATE-PROTON),'[M+C2H4O2-H]-':(1,1,ACETATE-PROTON),
 '[M+Cl]-':(1,1,MASS['Cl']+E),'[M]-':(1,1,E),
 '[M-2H]-':(1,2,-2*PROTON),'[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON),
 '[2M+H]+':(2,1,PROTON),'[2M+Na]+':(2,1,MASS['Na']-E),
 '[2M+NH4]+':(2,1,NH4-E),'[2M+K]+':(2,1,MASS['K']-E),
 '[2M-H]-':(2,1,-PROTON),'[2M+CH2O2-H]-':(2,1,FORMATE-PROTON),
 '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON),'[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
 '[3M+H]+':(3,1,PROTON),'[3M-H]-':(3,1,-PROTON)}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m]*z - d)/n
    return out
print('mass contract ready')

In [ ]:
import numpy as np, torch, time
G = 4
PPM = 10.0
# v9 caliber (validated, not v8): union 712,199 pool, single max-peak spec,
# 10ppm full-window nocap (no WMAX). Shipped window scored here; canonical
# max-merge -> top100 shortlist happens in the ICEBERG cell after RDKit install
# (collapse shipped->canon max, order (-score,canon), smi=best-shipped rep).
mids = list(mol.keys())
prows = []
for mid in mids:
    idxs = mol[mid]
    best = max(idxs, key=lambda i: len(TQ['ms2_mzs'][i]))
    prows.append((TQ['ms2_mzs'][best], TQ['ms2_normalized_intensities'][best], TQ['precursor_mz'][best],
            TQ['adduct'][best], TQ['instrument_type'][best], TQ['collision_energy_ev'][best], TQ['ionization_mode'][best]))
print('molecules:', len(mids))
@torch.no_grad()
def fwd_batch(rows):
    P = []
    for r in rows:
        a,b = prep_peaks(np.asarray(r[0],float), np.asarray(r[1],float), float(r[2]))
        P.append((a,b,r))
    Np = 128
    B = len(P)
    mz=np.zeros((B,Np),np.float32); it=np.zeros((B,Np),np.float32); pad=np.ones((B,Np),bool)
    for j,(a,b,_) in enumerate(P):
        n = min(len(a), Np)
        mz[j,:n]=a[:n]; it[j,:n]=b[:n]; pad[j,:n]=False
    T = lambda x: torch.as_tensor(x, device='cuda')
    rr = [r for _,_,r in P]
    args=(T(mz),T(it),T(pad),T(np.array([float(r[2]) for r in rr],np.float32)),
          T(np.array([ADDUCT_IX.get(r[3],ADDUCT_IX['<unk>']) for r in rr])),
          T(np.array([instr_family(r[4]) for r in rr])),
          T(np.array([float(np.mean(np.atleast_1d(r[5]))) if r[5] is not None else 25.0 for r in rr],np.float32)),
          T(np.array([1.0 if r[6]=='positive' else -1.0 for r in rr],np.float32)))
    return net(*args).float()
Z = torch.empty((len(prows), 6930))
BS = 64
for i in range(0, len(prows), BS):
    Z[i:i+BS] = fwd_batch(prows[i:i+BS]).cpu()
    if (i//BS+1) % 2 == 0: print('forward', min(i+BS, len(prows)), '/', len(prows))
print('all forwards done', tuple(Z.shape))
QM = neutral_mass(np.array([r[2] for r in prows], float), [r[3] for r in prows])
stat = {'empty_window':0,'mass_fallback':0,'cco':0,'exception':0}
print('forward+prep done', tuple(Z.shape))
NG = (len(mids) + G - 1)//G
print('groups:', NG, 'G=', G)
stat = {'empty_window':0,'mass_fallback':0,'cco':0,'exception':0}
PARTS = {}
def run_groups(g0, g1):
    import pickle
    tB = time.perf_counter()
    rows = []
    for g in range(g0*G, min(g1*G, len(mids)), G):
        try:
            chunk = range(g, min(g+G, len(mids)))
            Ws, KL, SL, PL = [], [], [], []
            for i in chunk:
                w = np.where(np.abs(allm - QM[i]) <= QM[i]*PPM*1e-6)[0]
                Ws.append(w)
                KL.append([allk[j] for j in w])
                SL.append([alls[j] for j in w])
                PL.append([abs(allm[j]-QM[i])/QM[i]*1e6 for j in w])
            L = max([len(w) for w in Ws] + [1])
            Fmat = np.zeros((len(chunk), L, 6930), np.float32)
            Fmat_np = np.ones((len(chunk), L), bool)
            for bi, w in enumerate(Ws):
                is_tr = w < NTR
                if is_tr.any():
                    Fmat[bi,np.where(is_tr)[0]] = np.unpackbits(np.asarray(t_fp[w[is_tr]], dtype=np.uint8), axis=1)[:, :6930]
                if (~is_tr).any():
                    ci = np.array([cidx[allk[j]] for j in w[~is_tr]])
                    Fmat[bi,np.where(~is_tr)[0]] = np.unpackbits(np.asarray(c_fp[ci], dtype=np.uint8), axis=1)[:, :6930]
                Fmat_np[bi,:len(w)] = False
            Fg = torch.as_tensor(Fmat, device='cuda')
            Zg = torch.as_tensor(Z[list(chunk)].numpy(), device='cuda').unsqueeze(-1)
            with torch.no_grad():
                S = torch.bmm(Fg, Zg).squeeze(-1)
            S[Fmat_np] = float('-inf')
            Scpu = S.cpu().numpy()
            for bi, i in enumerate(chunk):
                mid = mids[i]
                KLb, SLb, PLb = KL[bi], SL[bi], PL[bi]
                if not len(KLb):
                    stat['empty_window'] += 1
                    rows.append({'mid': mid, 'shipped': [], 'smiles': [], 'base_scores': [], 'n_window': 0, 'status': 'empty_window'})
                    continue
                sc = [float(x) for x in Scpu[bi][:len(KLb)].tolist()]
                pairs = sorted(zip(KLb, SLb, sc), key=lambda t: (-t[2], str(t[0])))
                if not pairs:
                    stat['mass_fallback'] += 1
                    ranked = sorted(zip(KLb, SLb, PLb), key=lambda t: t[2])
                    fb = [(k, sm, float(-pp)) for k, sm, pp in ranked]
                    rows.append({'mid': mid, 'shipped': [k for k, _, _ in fb], 'smiles': [s for _, s, _ in fb], 'base_scores': [x for _, _, x in fb], 'n_window': len(KLb), 'status': 'mass_fallback'})
                    continue
                rows.append({'mid': mid, 'shipped': [k for k, _, _ in pairs], 'smiles': [s for _, s, _ in pairs], 'base_scores': [x for _, _, x in pairs], 'n_window': len(KLb), 'status': 'ok'})
        except Exception as e:
            for i in chunk:
                rows.append({'mid': mids[i], 'shipped': [], 'smiles': [], 'base_scores': [], 'n_window': 0, 'status': 'exception'})
            stat['exception'] += len(chunk)
        if (g//G+1) % 5 == 0: print('group', g//G+1, round(time.perf_counter()-tB,1), stat)
    PARTS[g0] = rows
    pickle.dump({'records': rows}, open(f'/kaggle/working/partial_{g0}.pkl','wb'))
    print('block done', g0, g1, 'rows', len(rows), round(time.perf_counter()-tB,1), stat)


In [ ]:
run_groups(0, 20)

In [ ]:
run_groups(20, 40)

In [ ]:
run_groups(40, 60)

In [ ]:
run_groups(60, 80)

In [ ]:
run_groups(80, 100)

In [ ]:
import subprocess, sys, os, glob, json, time
t0 = time.time()
def find_first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    print(pattern, '->', len(hits))
    return hits
ICE_CANDS = find_first('/kaggle/input/**/ice_runner.py')
assert len(ICE_CANDS) >= 1, 'ice_runner.py not found under /kaggle/input: ' + str(sorted(os.listdir('/kaggle/input')))
ICE_PKG = os.path.dirname(ICE_CANDS[0])
print('ICE_PKG', ICE_PKG)
assert os.path.exists(os.path.join(ICE_PKG, 'fuse.py')), 'fuse.py missing'
# --- weight verification: msg_all mirror (NIST excluded), exact validated sizes ---
print('UPSTREAM msg_all preferred (ms-pred README rlkey=f1n6pbzx94g1k2el2wcbmee61); msg_simulation is the other folder; NIST EXCLUDED')
manifest = json.load(open(os.path.join(ICE_PKG, 'MANIFEST.json')))
print('MANIFEST model:', manifest.get('model'), '| ms_pred_commit:', manifest.get('ms_pred_commit'), '| rdkit pin:', manifest.get('rdkit'))
for rel, exp_bytes in [('ckpt/gen.pt', 14561244), ('ckpt/inten.pt', 13793037)]:
    p = os.path.join(ICE_PKG, rel)
    assert os.path.exists(p), f'missing weight {rel}'
    n = os.path.getsize(p)
    print(rel, 'bytes', n, 'expected', exp_bytes, 'OK' if n == exp_bytes else 'SIZE_MISMATCH')
    assert n == exp_bytes, f'size mismatch {rel}: {n} vs {exp_bytes}'
    exp = manifest['files'][rel]
    assert exp['bytes'] == exp_bytes, f'manifest bytes drift {rel}'
    import hashlib
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for ch in iter(lambda: f.read(1 << 20), b''):
            h.update(ch)
    got = h.hexdigest()
    print(rel, 'sha256', got[:16] + '...', 'OK' if got == exp['sha256'] else 'HASH_MISMATCH')
    assert got == exp['sha256'], f'sha256 mismatch {rel}'
print('weights verified: msg_all mirror gen.pt + inten.pt, NIST excluded')
WHL_FILES = find_first('/kaggle/input/**/*.whl')
print('whl files', len(WHL_FILES))
for p in WHL_FILES[:20]:
    print(p)
assert len(WHL_FILES) >= 9, f'wheelhouse incomplete: {len(WHL_FILES)}'
WHEEL_DIR = os.path.dirname(WHL_FILES[0])
print('WHEEL_DIR', WHEEL_DIR, sorted(os.listdir(WHEEL_DIR)))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps', '--find-links', WHEEL_DIR,
    'rdkit', 'einops', 'pytorch_lightning', 'lightning-utilities', 'torchmetrics', 'fsspec', 'typing_extensions', 'packaging', 'PyYAML'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-2000:])
import importlib
for m in ('rdkit', 'einops', 'pytorch_lightning', 'torchmetrics'):
    v = importlib.import_module(m)
    print(m, getattr(v, '__version__', 'ok'))
# --- assemble shipped windows -> canonical max-merge -> top100 shortlist (validated caliber; F1-optimal K) ---
all_records = []
for g0 in sorted(PARTS):
    all_records.extend(PARTS[g0])
print('assembled records', len(all_records), 'from blocks', sorted(PARTS))
rec_by_mid = {r['mid']: r for r in all_records}
assert len(rec_by_mid) == len(mids), f'records {len(rec_by_mid)} != mids {len(mids)}'
n_win_tot = sum(r.get('n_window', 0) for r in all_records)
print('total shipped window cands', n_win_tot, 'empty', sum(1 for r in all_records if not r['smiles']))
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
import rdkit as _rdk
print('rdkit (merge+export) =', _rdk.__version__)
_te = rdMolStandardize.TautomerEnumerator()
def _canon_key(smiles, fallback):
    try:
        m = Chem.MolFromSmiles(str(smiles))
        if m is None:
            return str(fallback), False
        Chem.RemoveStereochemistry(m)
        canon = _te.Canonicalize(m)
        key = Chem.MolToInchiKey(canon).split('-')[0]
        return str(key), True
    except Exception:
        return str(fallback), False
stat_merge = {'n_mols': len(mids), 'n_window_shipped': 0, 'n_canon_total': 0, 'canon_fallback': 0, 'empty': 0, 'n_top100_cands': 0}
CANON100 = {}
for mid in mids:
    rec = rec_by_mid[mid]
    sk = list(rec.get('shipped', [])); sm = list(rec.get('smiles', [])); bs = list(rec.get('base_scores', []))
    stat_merge['n_window_shipped'] += len(sk)
    if not sk:
        stat_merge['empty'] += 1
        CANON100[mid] = {'canon': [], 'smiles': [], 'base': [], 'n_window': 0, 'status': rec.get('status', 'empty_window')}
        continue
    best = {}
    for k, s, b in zip(sk, sm, bs):
        ck, ok = _canon_key(s, k)
        if not ok:
            stat_merge['canon_fallback'] += 1
        prev = best.get(ck)
        if prev is None or float(b) > prev[2]:
            best[ck] = (k, str(s), float(b))
    cks = sorted(best.keys())
    ordered = sorted(cks, key=lambda c: (-best[c][2], c))
    top = ordered[:100]
    CANON100[mid] = {'canon': top, 'smiles': [best[c][1] for c in top], 'base': [best[c][2] for c in top], 'n_window': len(sk), 'n_canon': len(cks), 'status': 'ok'}
    stat_merge['n_canon_total'] += len(cks)
    stat_merge['n_top100_cands'] += len(top)
print('merge stat', stat_merge)
assert len(CANON100) == len(mids)
# --- build ICE items: top100 cands + ALL spectra per molecule (runner filters covered [M+H]+/[M+Na]+ positive) ---
items = []
for mid in mids:
    c100 = CANON100[mid]
    cands = list(c100['smiles'][:100])
    canons = list(c100['canon'][:100])
    bases = list(c100['base'][:100])
    idxs = mol[mid]
    specs = []
    for ii in idxs:
        ce_ev = TQ['collision_energy_ev'][ii]
        specs.append({
            'mz': list(map(float, TQ['ms2_mzs'][ii])),
            'it': list(map(float, TQ['ms2_normalized_intensities'][ii])),
            'prec': float(TQ['precursor_mz'][ii]),
            'adduct': str(TQ['adduct'][ii]),
            'mode': str(TQ['ionization_mode'][ii]),
            'instrument': str(TQ['instrument_type'][ii]),
            'ce_ev': list(map(float, ce_ev)) if ce_ev is not None else [],
            'ce_orig': str(TQ['collision_energy_orig'][ii]) if TQ['collision_energy_orig'][ii] is not None else '',
            'ce_units': str(TQ['collision_energy_orig_units'][ii]) if TQ['collision_energy_orig_units'][ii] is not None else '',
        })
    items.append({'mid': mid, 'cands': cands, 'canons': canons, 'base': bases, 'spectra': specs})
print('items', len(items), 'total cands(top100)', sum(len(x['cands']) for x in items), 'total spectra', sum(len(x['spectra']) for x in items))
assert all(all(isinstance(_b, float) and _b == _b and abs(_b) != float('inf') for _b in _it['base']) for _it in items), 'non-finite base score'
with open('/kaggle/working/canon100.json', 'w') as f:
    json.dump({m: {'canon': CANON100[m]['canon'], 'smiles': CANON100[m]['smiles'], 'base': CANON100[m]['base']} for m in mids}, f)
import math, hashlib, re, gc, importlib.util
# --- F1-split forward: per-record CE expansion + ICEBERG prediction (F1e-optimal: top100 + fused_merge) ---
# Budget: top100 split = 43,707 unique jobs on NP250 at ~26-30/s -> ~25-30 min + load/overhead;
# forward budget_s = 3600 (BUDGET_S); total far below 9h. Single-CE spectra keep the original path;
# multi-CE covered spectra get one prediction per CE. The NCE branch implements runner semantics and is
# counted (dormant on all-eV input as on NP250). Two fusions stored per candidate: sims-mean (mean of the
# per-CE similarities, auxiliary) and fused_merge (equal-weight merged predictions then similarity,
# primary). Missing stays None, never 0. Scoring primitives (merge/prep_query/entropy_sim/is_covered/
# spectrum_ce/ce_bucket/instr_token/prep_spectrum/Predictor) are the runner's own, as in f1_worker.py.
BUDGET_S = 3600.0
PREP_TAG_F1 = ('panel-A1:v1(clean0.001/512/prec+2>x2->prep0.002/256/p1/entw;'
               'sim0.01Da/20ppm)')
def _peak_hash(mz, it):
    h = hashlib.sha256()
    h.update(np.round(np.asarray(mz, np.float64), 4).astype('<f4').tobytes())
    h.update(np.asarray(it, np.float64).astype('<f4').tobytes())
    return h.hexdigest()[:16]
def _cache_key(model, ckpt_sha, structure, ce_bucket, adduct, instr):
    ceb = 'sum' if (isinstance(ce_bucket, float) and math.isnan(ce_bucket)) else ('%.1f' % float(ce_bucket))
    parts = [model, ckpt_sha, str(structure), ceb, str(adduct), str(instr), PREP_TAG_F1]
    return hashlib.sha256('|'.join(parts).encode()).hexdigest()[:16], parts
def _parse_ce_numbers(s):
    return [float(x) for x in re.findall(r'[0-9]+[.]?[0-9]*', str(s or ''))]
def _jnum(x):
    try:
        f = float(x)
    except Exception:
        return None
    return f if math.isfinite(f) else None
def _expand_records(R, spectra):
    recs = []
    for qi, sp in enumerate(spectra):
        sp = dict(sp)
        covered = bool(R.is_covered(sp))
        units = sp.get('ce_units')
        nce = (units == 'NCE')
        ce_ev = [float(v) for v in (sp.get('ce_ev') or [])]
        if not covered:
            recs.append({'qid': qi, 'sp': sp, 'ces': [], 'ce_source': 'uncovered', 'nce': bool(nce), 'covered': False})
            continue
        if len(ce_ev) <= 1:
            recs.append({'qid': qi, 'sp': sp, 'ces': [float(R.spectrum_ce(sp))], 'ce_source': 'single', 'nce': bool(nce), 'covered': True})
            continue
        if nce:
            nums = _parse_ce_numbers(sp.get('ce_orig'))
            if len(nums) == len(ce_ev):
                ces, src = list(nums), 'nce-split'
            else:
                ces, src = [float(R.spectrum_ce(sp))], 'nce-mean-fallback'
        else:
            ces, src = list(ce_ev), 'ev-split'
        recs.append({'qid': qi, 'sp': sp, 'ces': ces, 'ce_source': src, 'nce': bool(nce), 'covered': True})
    return recs
def _cond_entries(R, recs):
    entries = []
    for r in recs:
        if not r['covered']:
            continue
        qm, qp = R.prep_spectrum(r['sp'])
        ad = r['sp']['adduct']
        ins = R.instr_token(r['sp'])
        for ce in r['ces']:
            ceb = R.ce_bucket(ce)
            entries.append({'qid': r['qid'], 'qm': qm, 'qp': qp, 'adduct': ad, 'instr': ins,
                            'ce': float(ce) if ce == ce else float('nan'), 'ce_bucket': ceb,
                            'ce_source': r['ce_source'], 'nce': r['nce']})
    return entries
def _score_simsmean(R, entries, cands, cache):
    fused, detail = {}, {}
    sum_cache = {}
    for c in cands:
        sims, recs = [], []
        for e in entries:
            ceb = e['ce_bucket']
            if isinstance(ceb, float) and math.isnan(ceb):
                sk = (c, e['adduct'], e['instr'])
                if sk not in sum_cache:
                    parts = [cache.get((c, e['adduct'], e['instr'], ce)) for ce in R.SUM_CE]
                    parts = [p for p in parts if p is not None]
                    sum_cache[sk] = R.merge(parts) if len(parts) == len(R.SUM_CE) else None
                pr, how = sum_cache[sk], 'merged-sum'
            else:
                pr, how = cache.get((c, e['adduct'], e['instr'], ceb)), 'single'
            if pr is None:
                continue
            pm, pp = R.prep_query(pr[0], pr[1], R.FLOOR, R.TOPK, R.POWER, R.ENTW)
            s = R.entropy_sim(e['qm'], e['qp'], pm, pp, R.TOL, R.PPM)
            if math.isfinite(float(s)):
                sims.append(float(s))
                recs.append({'qid': e['qid'], 'adduct': e['adduct'], 'instr': e['instr'],
                             'ce': _jnum(e['ce']), 'ce_bucket': (None if (isinstance(ceb, float) and math.isnan(ceb)) else float(ceb)),
                             'ce_source': e['ce_source'], 'nce': bool(e['nce']), 'how': how, 'sim': float(s)})
        fused[c] = float(np.mean(sims)) if sims else None
        detail[c] = recs
    return fused, detail
def _score_merged(R, recs, entries_by_qid, cands, cache):
    fused, detail = {}, {}
    for c in cands:
        sims, out = [], []
        for r in recs:
            if not r['covered']:
                continue
            ad, ins = r['sp']['adduct'], R.instr_token(r['sp'])
            parts, ok = [], True
            for ce in r['ces']:
                ceb = R.ce_bucket(ce)
                if isinstance(ceb, float) and math.isnan(ceb):
                    ok = False
                    break
                pr = cache.get((c, ad, ins, ceb))
                if pr is None:
                    ok = False
                    break
                parts.append(pr)
            if not ok or not parts:
                continue
            merged = R.merge(parts) if len(parts) > 1 else parts[0]
            qe = entries_by_qid[r['qid']]
            pm, pp = R.prep_query(merged[0], merged[1], R.FLOOR, R.TOPK, R.POWER, R.ENTW)
            s = R.entropy_sim(qe['qm'], qe['qp'], pm, pp, R.TOL, R.PPM)
            if math.isfinite(float(s)):
                sims.append(float(s))
                out.append({'qid': r['qid'], 'adduct': ad, 'instr': ins,
                            'ce': [_jnum(v) for v in r['ces']], 'ce_source': r['ce_source'],
                            'nce': bool(r['nce']), 'how': 'merged-pred', 'sim': float(s)})
        fused[c] = float(np.mean(sims)) if sims else None
        detail[c] = out
    return fused, detail
F1_SCORES = '/kaggle/working/iceberg_scores_f1.json'
F1_META = '/kaggle/working/ice_meta_f1.json'
t1 = time.time()
deadline = t1 + BUDGET_S
_spec = importlib.util.spec_from_file_location('v9_ice_runner', os.path.join(ICE_PKG, 'ice_runner.py'))
R = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(R)
class _FA:
    pass
_fa = _FA()
_fa.pkg = ICE_PKG
_fa.site = 'none'
_fa.wheels = None
_fa.any_rdkit = True
rdkit_v9ice = R.setup_env(_fa, print)
print('ice env rdkit =', rdkit_v9ice, '(recorded as-is)')
gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass
def _sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for ch in iter(lambda: f.read(1 << 20), b''):
            h.update(ch)
    return h.hexdigest()
CK_SHORT = _sha_file(os.path.join(ICE_PKG, 'ckpt', 'gen.pt'))[:8] + _sha_file(os.path.join(ICE_PKG, 'ckpt', 'inten.pt'))[:8]
print('ckpt short', CK_SHORT)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'need GPU for ICE forward'
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
ice_model = R.load_model(ICE_PKG, dev, print)
pred = R.Predictor(ice_model, dev, 96, print)
print('ICEBERG predictor ready on', dev)
okc = {}
for _it in items:
    for _c in _it['cands']:
        if _c not in okc:
            okc[_c], _why = R.smiles_ok(_c)
            if not okc[_c] and sum(1 for _v in okc.values() if not _v) <= 5:
                print('smiles rejected (%s): %s' % (_why, str(_c)[:100]))
print('smiles ok:', sum(1 for _v in okc.values() if _v), 'bad:', sum(1 for _v in okc.values() if not _v))
plans = {}
nce_hits, single_hits, split_hits, uncovered_hits = 0, 0, 0, 0
for _it in items:
    try:
        _recs = _expand_records(R, _it['spectra'])
        _entries = _cond_entries(R, _recs)
        plans[_it['mid']] = (_recs, _entries)
        for _r in _recs:
            if not _r['covered']:
                uncovered_hits += 1
            elif _r['nce']:
                nce_hits += 1
            elif _r['ce_source'] == 'single':
                single_hits += 1
            else:
                split_hits += 1
    except Exception as _e:
        print('plan failed', _it['mid'], repr(_e)[:200])
        plans[_it['mid']] = ([], [])
n_covered = sum(1 for _v in plans.values() if _v[1])
print('mols covered:', n_covered, '/', len(items), '| single:', single_hits, 'split:', split_hits, 'nce:', nce_hits, 'uncovered:', uncovered_hits)
uq = set()
for _it in items:
    for _e in plans[_it['mid']][1]:
        _ceb = _e['ce_bucket']
        _ck = 'sum' if (isinstance(_ceb, float) and math.isnan(_ceb)) else round(float(_ceb), 1)
        for _c in _it['cands']:
            if okc.get(_c):
                uq.add((_c, _e['adduct'], _e['instr'], _ck))
print('unique (cand,adduct,instr,CE) jobs:', len(uq))
def _mol_jobs(_it):
    _recs, _entries = plans[_it['mid']]
    J = set()
    for _e in _entries:
        _ceb = _e['ce_bucket']
        for _c in _it['cands']:
            if not okc.get(_c):
                continue
            if isinstance(_ceb, float) and math.isnan(_ceb):
                for _ce in R.SUM_CE:
                    J.add((_c, _e['adduct'], _e['instr'], _ce))
            else:
                J.add((_c, _e['adduct'], _e['instr'], _ceb))
    return J
def _score_mol(_it):
    _mid = _it['mid']
    _recs, _entries = plans[_mid]
    _cands = [_c for _c in _it['cands'] if okc.get(_c)]
    _fused, _detail = _score_simsmean(R, _entries, _cands, pred.cache)
    _fused_m, _detail_m = _score_merged(R, _recs, {_e['qid']: _e for _e in _entries}, _cands, pred.cache)
    _per = {}
    for _ci, _c in enumerate(_it['cands']):
        if not okc.get(_c):
            continue
        _peaks = {}
        for _r in _detail.get(_c, []):
            if _r['how'] == 'merged-sum':
                _cbs = [float(_x) for _x in R.SUM_CE]
            else:
                _cbs = [(float('nan') if _r['ce_bucket'] is None else float(_r['ce_bucket']))]
            _hs = []
            for _cb in _cbs:
                _hk, _p = _cache_key('iceberg', CK_SHORT, _c, _cb, _r['adduct'], _r['instr'])
                _hs.append(_hk)
            _peaks['%s|%s|%s|%s' % (_r['qid'], _r['adduct'], _r['instr'], _r['ce_bucket'])] = (_hs[0] if len(_hs) == 1 else _hs)
        _per[_c] = {'canon': _it['canons'][_ci], 'base': _it['base'][_ci],
                    'fused': _fused.get(_c), 'sims': _detail.get(_c, []), 'peaks': _peaks,
                    'fused_merge': _fused_m.get(_c), 'sims_merge': _detail_m.get(_c, [])}
    return _per
f1_raw = {}
f1_meta = {'model': 'iceberg', 'cemode': 'split', 'k': 100, 'status': 'init', 'n_mols': len(items),
           'n_mols_covered': n_covered, 'n_mols_scored': 0, 'n_cands': sum(len(_it['cands']) for _it in items),
           'n_scored_merge': 0, 'n_scored_mean': 0, 'rdkit': rdkit_v9ice, 'device': dev,
           'ckpt_short': CK_SHORT, 'prep_tag': PREP_TAG_F1, 'nce_records': nce_hits,
           'single_records': single_hits, 'split_records': split_hits, 'uncovered_records': uncovered_hits,
           'unique_jobs': len(uq), 'budget_s': BUDGET_S, 'errors': []}
def _write_ckpt():
    with open(F1_SCORES, 'w') as _f:
        json.dump(f1_raw, _f, allow_nan=False)
_write_ckpt()
def _flush_chunk(chunk, chunk_jobs):
    t0 = time.time()
    finished = pred.predict(chunk_jobs, deadline)
    n_new = 0
    if finished:
        for _cit in chunk:
            try:
                f1_raw[_cit['mid']] = _score_mol(_cit)
                n_new += 1
            except Exception as _e:
                f1_meta['errors'].append('%s: score %r' % (_cit['mid'], _e))
        _write_ckpt()
        print('chunk done: %d mols %d jobs %.1fs (covered scored %d/%d)' % (len(chunk), len(chunk_jobs), time.time() - t0, f1_meta['n_mols_scored'] + n_new, n_covered))
    f1_meta['n_mols_scored'] += n_new
    return finished
chunk, chunk_jobs, stopped = [], set(), False
for _it in items:
    if not plans[_it['mid']][1]:
        f1_raw[_it['mid']] = {_c: {'canon': _it['canons'][_ci], 'base': _it['base'][_ci], 'fused': None, 'sims': [], 'peaks': {}, 'fused_merge': None, 'sims_merge': [], 'note': 'no covered spectrum'} for _ci, _c in enumerate(_it['cands'])}
        continue
    chunk.append(_it)
    chunk_jobs |= _mol_jobs(_it)
    if len(chunk_jobs) >= 3000:
        if not _flush_chunk(chunk, chunk_jobs):
            stopped = True
            chunk, chunk_jobs = [], set()
            break
        chunk, chunk_jobs = [], set()
if not stopped and chunk:
    if not _flush_chunk(chunk, chunk_jobs):
        stopped = True
    chunk, chunk_jobs = [], set()
for _it in items:
    if _it['mid'] not in f1_raw:
        f1_raw[_it['mid']] = {_c: {'canon': _it['canons'][_ci], 'base': _it['base'][_ci], 'fused': None, 'sims': [], 'peaks': {}, 'fused_merge': None, 'sims_merge': [], 'note': 'not reached (budget)'} for _ci, _c in enumerate(_it['cands'])}
_write_ckpt()
f1_meta.update(status=('budget' if stopped else 'ok'), seconds=round(time.time() - t1, 1),
               n_pred=pred.n_pred, n_pred_fail=getattr(pred, 'n_fail', 0),
               t_pred=round(getattr(pred, 't_pred', 0.0), 2),
               pred_per_s=round(pred.n_pred / max(getattr(pred, 't_pred', 0.0), 1e-9), 2))
try:
    f1_meta['max_mem_gb'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)
except Exception:
    pass
f1_meta['n_scored_merge'] = sum(1 for _v in f1_raw.values() for _r in _v.values() if _r.get('fused_merge') is not None)
f1_meta['n_scored_mean'] = sum(1 for _v in f1_raw.values() for _r in _v.values() if _r.get('fused') is not None)
with open(F1_META, 'w') as _f:
    json.dump(f1_meta, _f, allow_nan=False)
print(json.dumps({k: f1_meta.get(k) for k in ('status', 'n_mols', 'n_mols_covered', 'n_mols_scored', 'n_cands', 'n_scored_merge', 'n_scored_mean', 'seconds', 'pred_per_s', 'device', 'rdkit', 'max_mem_gb', 'unique_jobs', 'nce_records')}, indent=1))
print('wrote', F1_SCORES, round(os.path.getsize(F1_SCORES) / 1024, 1), 'KB')
assert f1_meta['status'] == 'ok', 'forward incomplete: %s %s' % (f1_meta['status'], f1_meta['errors'][-3:])
assert isinstance(f1_raw, dict) and set(f1_raw) == set(mids), 'ice output incomplete'


In [ ]:
import math, json, csv, os, pickle
def _z(vals):
    ok = [v is not None and isinstance(v, (int, float)) and math.isfinite(v) for v in vals]
    xs = [float(v) for v, o in zip(vals, ok) if o]
    n = len(xs)
    if n < 2:
        return [0.0] * len(vals)
    mu = sum(xs) / n
    sd = math.sqrt(sum((x - mu) ** 2 for x in xs) / (n - 1))
    if not sd > 0:
        return [0.0] * len(vals)
    return [((float(v) - mu) / sd) if o else 0.0 for v, o in zip(vals, ok)]
try:
    import rdkit
    print('rdkit =', rdkit.__version__)
    from rdkit import Chem
    HAVE_RDKIT = True
except Exception as e:
    print('rdkit import failed', repr(e)[:500])
    HAVE_RDKIT = False
    Chem = None
# v9-F1: fuse over canonical top100 shortlist (CANON100 from ICE cell); both forward scores stored, merge-ranked
try:
    CANON100
except NameError:
    CANON100 = {m: None for m in mids}
    _cj = json.load(open('/kaggle/working/canon100.json'))
    for m in mids:
        CANON100[m] = {'canon': _cj[m]['canon'], 'smiles': _cj[m]['smiles'], 'base': _cj[m]['base']}
print('fuse shortlist source: CANON100 mols', len(CANON100), 'empty', sum(1 for m in mids if not CANON100[m]['smiles']))
ice_f1 = json.load(open('/kaggle/working/iceberg_scores_f1.json'))
assert isinstance(ice_f1, dict) and set(ice_f1) == set(mids), 'iceberg_scores_f1 incomplete'
f1_meta = json.load(open('/kaggle/working/ice_meta_f1.json'))
print('forward meta:', json.dumps({k: f1_meta.get(k) for k in ('status', 'n_mols_covered', 'n_mols_scored', 'n_scored_merge', 'n_scored_mean', 'pred_per_s', 'unique_jobs', 'nce_records')}, indent=1))
out_rows = []
stat_fuse = {'cco_empty': 0, 'pad_cco': 0, 'raw_fallback': 0, 'raw_fallback_no_rdkit': 0, 'n_scored_mols_merge': 0, 'n_scored_cands_merge': 0, 'n_scored_cands_mean': 0, 'n_total_cands': 0, 'top1_agree_merge_mean': 0, 'n_mols_both_scored': 0}
def _pick(rec, key):
    v = rec.get(key) if isinstance(rec, dict) else None
    return float(v) if (v is not None and isinstance(v, (int, float)) and math.isfinite(float(v))) else None
for mid in mids:
    c100 = CANON100[mid]
    smiles = list(c100['smiles'])
    base = list(c100['base'])
    n = len(smiles)
    stat_fuse['n_total_cands'] += n
    if n == 0:
        out_rows.append((mid, ';'.join(['CCO']*25)))
        stat_fuse['cco_empty'] += 1
        continue
    gdict = ice_f1.get(mid, {})
    for _sm in smiles:
        _rec = gdict.get(_sm, {})
        if isinstance(_rec, dict):
            for _k in ('fused_merge', 'fused'):
                _vv = _rec.get(_k)
                assert _vv is None or (isinstance(_vv, (int, float)) and math.isfinite(float(_vv))), 'stored score must be None or finite'
    iv_merge = [_pick(gdict.get(sm, {}), 'fused_merge') for sm in smiles]
    iv_mean = [_pick(gdict.get(sm, {}), 'fused') for sm in smiles]
    n_sc = sum(1 for v in iv_merge if v is not None)
    stat_fuse['n_scored_cands_merge'] += n_sc
    stat_fuse['n_scored_cands_mean'] += sum(1 for v in iv_mean if v is not None)
    if n_sc > 0:
        stat_fuse['n_scored_mols_merge'] += 1
    zb = _z(base)
    zm = _z(iv_merge)
    fused = [a + b for a, b in zip(zb, zm)]
    scored_idx = [i for i, v in enumerate(iv_merge) if v is not None]
    sset = set(scored_idx)
    unscored_idx = [i for i in range(n) if i not in sset]
    scored_sorted = sorted(scored_idx, key=lambda i: (-fused[i], i))
    unscored_sorted = sorted(unscored_idx, key=lambda i: (-base[i], i))
    full_order = scored_sorted + unscored_sorted
    za = _z(iv_mean)
    fused_aux = [a + b for a, b in zip(zb, za)]
    scored_aux = sorted([i for i, v in enumerate(iv_mean) if v is not None], key=lambda i: (-fused_aux[i], i))
    unscored_aux = sorted([i for i in range(n) if iv_mean[i] is None], key=lambda i: (-base[i], i))
    aux_order = scored_aux + unscored_aux
    if sum(1 for v in iv_merge if v is not None) > 0 and sum(1 for v in iv_mean if v is not None) > 0:
        stat_fuse['n_mols_both_scored'] += 1
        if full_order[0] == aux_order[0]:
            stat_fuse['top1_agree_merge_mean'] += 1
    top25_idx = full_order[:25]
    top25_raw = [smiles[i] for i in top25_idx]
    if len(top25_raw) < 25:
        top25_raw = top25_raw + ['CCO'] * (25 - len(top25_raw))
        stat_fuse['pad_cco'] += 1
    canon_out = []
    for sm in top25_raw:
        if sm == 'CCO':
            canon_out.append('CCO')
            continue
        if not HAVE_RDKIT:
            canon_out.append(sm)
            stat_fuse['raw_fallback_no_rdkit'] += 1
            continue
        try:
            m = Chem.MolFromSmiles(sm)
            if m is None:
                canon_out.append(sm)
                stat_fuse['raw_fallback'] += 1
            else:
                canon_out.append(Chem.MolToSmiles(m))
        except Exception:
            canon_out.append(sm)
            stat_fuse['raw_fallback'] += 1
    out_rows.append((mid, ';'.join(canon_out)))
print('fused rows', len(out_rows), stat_fuse)
print('blocks stat', stat)
with open('/kaggle/working/submission.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['molecule_id','smiles'])
    w.writerows(out_rows)
ncco_first = sum(1 for _, s in out_rows if s.split(';')[0] == 'CCO')
print('wrote', len(out_rows), 'bare_CCO_first', ncco_first)
print('coverage blocks', stat, 'fuse', stat_fuse)
assert len(out_rows)==len(mids)
for _, s in out_rows:
    assert len(s.split(';'))==25, 'row must have 25 cands'
print('verified 400x25' if len(out_rows)==400 else f'verified {len(out_rows)}x25')

产物 `/kaggle/working/submission.csv`（v9：FPNet single 10ppm nocap canonical top100 + ICEBERG F1-split fused_merge为主/sims-mean为辅 z-融合 top25，RDKit 规范导出）。覆盖率随行打印；CCO 仅空窗/异常/补位时出现；RDKit 版本如实打印；权重为 msg_all 镜像（NIST excluded）。